In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os

# Load Dataset

In [14]:
# df = pd.read_csv("../data/sintetis/dataset_irigasi.csv")
df = pd.read_csv("../data/dataset_irigasi.csv")
print(f"Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Dataset: 255 baris, 4 kolom


,soil_moisture,air_temperature,air_humidity,irrigation_action
0,23.7,24.4,75.6,0
1,23.5,24.4,75.5,0
2,23.3,24.4,75.9,0
3,23.2,24.5,76.2,0
4,23.0,24.5,76.0,0


# Pisahkan Fitur & Label

In [15]:
FEATURES = ["soil_moisture", "air_temperature", "air_humidity"]

X = df[FEATURES].values
y = df["irrigation_action"].values

print(f"Fitur: {FEATURES}")
print(f"Label 0 (tidak siram): {(y==0).sum()}")
print(f"Label 1 (siram):       {(y==1).sum()}")
print(f"Rasio siram: {y.mean()*100:.1f}%")

Fitur: ['soil_moisture', 'air_temperature', 'air_humidity']
Label 0 (tidak siram): 161
Label 1 (siram):       94
Rasio siram: 36.9%


In [16]:
if len(set(y)) < 2:
    print("⚠️ BAHAYA: cuma 1 kelas! Data belum variatif, model gak bisa dilatih.")
else:
    print(f"OK — {len(set(y))} kelas. Lanjut.")

OK — 2 kelas. Lanjut.


# Split Train/Test

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # removed stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")


Train: 204, Test: 51


# Training Random Forest

In [18]:
model = RandomForestClassifier(
    n_estimators=15,     
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Training selesai.")

Training selesai.


# Cross Validation

In [19]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
print(f"CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV F1: 0.9377 (+/- 0.0313)


# Evaluasi di Test Set

In [20]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Tidak siram', 'Siram'])}")

Accuracy: 0.9608
F1: 0.9474

Confusion Matrix:
[[31  2]
 [ 0 18]]

              precision    recall  f1-score   support

 Tidak siram       1.00      0.94      0.97        33
       Siram       0.90      1.00      0.95        18

    accuracy                           0.96        51
   macro avg       0.95      0.97      0.96        51
weighted avg       0.96      0.96      0.96        51



# Feature importance

In [21]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print("Feature Importance:")
for idx in sorted_idx:
    bar = "█" * int(importances[idx] * 50)
    print(f"  {FEATURES[idx]:18s} {importances[idx]:.4f}  {bar}")

Feature Importance:
  air_temperature    0.4657  ███████████████████████
  soil_moisture      0.4075  ████████████████████
  air_humidity       0.1268  ██████


# Test Prediksi Manual

In [ ]:
# --- Test case: (deskripsi, input row, label yang DIHARAPKAN) ---
kasus_uji = [
    # deskripsi                          soil_moisture, air_temp, air_humidity, ec, harapan
    ("Sebelum siram PAGI (acuan petani)",   11.0, 23.9, 71.9, 3962,  1),  # kering pagi -> siram
    ("Sebelum siram JAM 11 (panas+kering)", 25.0, 32.9, 42.0, 3915,  1),  # panas -> ambang naik -> siram
    ("Baru disiram (basah)",                36.5,  9.4, 24.7, 3694,  0),  # basah -> jangan
    ("Zona aman siang normal",              22.0, 28.0, 60.0, 3850,  0),  # cukup -> jangan
    ("Sensor mati (0)",                      0.0,  0.0, 25.8, 0,    -1),  # rusak -> buang
    ("Spike jenuh (>45)",                   74.6, 14.0, 24.3, 3293, -1),  # rusak -> buang
    ("Malam sangat lembap, agak kering",    12.5, 16.0, 90.0, 3970,  1),  # ah>85 -> ambang turun -> siram
]

def buat_row(sm, at, ah, ec):
    return {"soil_moisture": sm, "air_temperature": at,
            "air_humidity": ah, "ec": ec}

print(f"{'Deskripsi':<38}{'sm':>6}{'hasil':>7}{'harap':>7}  status")
print("-" * 72)

lulus = 0
for desk, sm, at, ah, ec, harap in kasus_uji:
    hasil = label_irigasi(buat_row(sm, at, ah, ec))
    ok = "OK" if hasil == harap else "XX GAGAL"
    if hasil == harap:
        lulus += 1
    print(f"{desk:<38}{sm:>6}{hasil:>7}{harap:>7}  {ok}")

print("-" * 72)
print(f"Lulus: {lulus}/{len(kasus_uji)}")


Deskripsi                                 sm  hasil  harap  status
------------------------------------------------------------------------
Sebelum siram PAGI (acuan petani)       11.0      1      1  OK
Sebelum siram JAM 11 (panas+kering)     25.0      0      1  XX GAGAL
Baru disiram (basah)                    36.5      0      0  OK
Zona aman siang normal                  22.0      0      0  OK
Sensor mati (0)                          0.0     -1     -1  OK
Spike jenuh (>45)                       74.6      0     -1  XX GAGAL
Malam sangat lembap, agak kering        12.5      0      1  XX GAGAL
------------------------------------------------------------------------
Lulus: 4/7


# Simpan Model

In [23]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/rf_irigasi.joblib")
print("Model tersimpan: models/rf_irigasi.joblib")

Model tersimpan: models/rf_irigasi.joblib


# Convert ke esp

In [24]:
from micromlgen import port

c_code = port(model)
with open("model_irigasi.h", "w") as f:
    f.write(c_code)
print("Tersimpan: model_irigasi.h")
print(f"Ukuran: {len(c_code)} karakter")

Tersimpan: model_irigasi.h
Ukuran: 24478 karakter
